In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import re
import joblib
import string

In [2]:
fake = pd.read_csv('Fake.csv')
true = pd.read_csv('True.csv')

In [3]:
fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [4]:
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [5]:
fake['class']=0
true['class']=1

In [6]:
data = pd.concat([fake, true], axis=0)

In [7]:
data.sample(10)

,title,text,subject,date,class
15677,A BRUTALLY HONEST MESSAGE TO BLACK AMERICAN ME...,****STRONG LANGUAGE Warning****You have lit a ...,politics,"May 18, 2015",0
14307,BLACK PASTOR BRINGS DOWN THE HOUSE At Trump Ra...,Only God could take a black guy from the inne...,politics,"Mar 13, 2016",0
2261,Senior U.S. envoy tries to calm fears over Sta...,WASHINGTON (Reuters) - The No. 2 U.S. diplomat...,politicsNews,"August 8, 2017",1
3476,Israel disappointed at Trump's decision not to...,JERUSALEM (Reuters) - Israel on Thursday expre...,politicsNews,"June 1, 2017",1
14567,OBAMA THROWS HILLARY UNDER THE BUS: E-mail Sca...,Obama decides to throw Hillary under the bus I...,politics,"Jan 30, 2016",0
10370,New disturbance at Trump rally as crucial Repu...,"DAYTON, Ohio (Reuters) - Secret Service office...",politicsNews,"March 12, 2016",1
14399,U.N. urges Japan to resettle more refugees aft...,TOKYO (Reuters) - The U.N. refugee agency has ...,worldnews,"November 20, 2017",1
17973,Referendum likely on Dutch 'tapping' law,AMSTERDAM (Reuters) - Digital privacy advocate...,worldnews,"October 9, 2017",1
2814,Senate may vote on revised healthcare bill nex...,WASHINGTON (Reuters) - U.S. Senate Republicans...,politicsNews,"July 11, 2017",1
13810,JIMMY KIMMEL READS His Second Book To Donald T...,Jimmy Kimmel writes a second book for Trump pr...,politics,"May 27, 2016",0


In [8]:
data = data.drop(['title','subject','date'],axis=1)

In [9]:
data.reset_index(inplace=True)

In [10]:
data.drop(['index'],axis=1,inplace=True)

In [11]:
data.sample(5)

,text,class
37749,COPENHAGEN (Reuters) - Denmark intends to inve...,1
28851,NEW YORK/LOS ANGELES (Reuters) - A second cons...,1
22050,Tune in to the Alternate Current Radio Network...,0
17764,This is a truly disturbing discovery and an un...,0
38842,BERLIN (Reuters) - Germany s highest court rul...,1


In [12]:
def clean_text(text):
    text = text.lower() 
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('\\W', ' ', text)
    text = re.sub('http\S+|www\S+|https\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

<>:3: SyntaxWarning: invalid escape sequence '\['
<>:5: SyntaxWarning: invalid escape sequence '\S'
<>:9: SyntaxWarning: invalid escape sequence '\w'
<>:3: SyntaxWarning: invalid escape sequence '\['
<>:5: SyntaxWarning: invalid escape sequence '\S'
<>:9: SyntaxWarning: invalid escape sequence '\w'
C:\Users\kavinkumar\AppData\Local\Temp\ipykernel_25968\988964414.py:3: SyntaxWarning: invalid escape sequence '\['
  text = re.sub('\[.*?\]', '', text)
C:\Users\kavinkumar\AppData\Local\Temp\ipykernel_25968\988964414.py:5: SyntaxWarning: invalid escape sequence '\S'
  text = re.sub('http\S+|www\S+|https\S+', '', text)
C:\Users\kavinkumar\AppData\Local\Temp\ipykernel_25968\988964414.py:9: SyntaxWarning: invalid escape sequence '\w'
  text = re.sub('\w*\d\w*', '', text)


In [13]:
data['text'] = data['text'].apply(clean_text)

In [14]:
X = data['text']
y = data['class']

xtrain, xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=42)   

In [15]:
# Ensure text data are strings (avoid sparse-matrix/bytes issues)
xtrain = xtrain.astype(str)
xtest = xtest.astype(str)

vectorizer = TfidfVectorizer()
xv_train = vectorizer.fit_transform(xtrain)
xv_test = vectorizer.transform(xtest)

# Save the vectorizer for later use
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
print('Vectorizer fitted and saved to tfidf_vectorizer.pkl')

Vectorizer fitted and saved to tfidf_vectorizer.pkl


In [16]:
from sklearn.metrics import accuracy_score

# Train logistic regression (increase max_iter to ensure convergence)
lr = LogisticRegression(max_iter=1000)
lr.fit(xv_train, ytrain)

# Evaluate
pred = lr.predict(xv_test)
print('Classification report:')
print(classification_report(ytest, pred))
print('Accuracy:', accuracy_score(ytest, pred))

# Save the trained model
joblib.dump(lr, 'logistic_model.pkl')
print('Trained model saved to logistic_model.pkl')

Classification report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      4733
           1       0.98      0.99      0.98      4247

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980

Accuracy: 0.9856347438752784
Trained model saved to logistic_model.pkl
